# DistilBERT — Consumer Complaint Classification


## Step 0 — Install packages and verify GPU

In [ ]:
!pip install -q transformers datasets accelerate

import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("\n!! NO GPU DETECTED !!")
    print("Go to Runtime > Change runtime type > T4 GPU, then re-run this cell.")
    print("Do not continue without a GPU — training will take many hours.")

## Step 2 — Load and inspect

Confirms the files match what the Logistic Regression model used.

In [ ]:
import pandas as pd
import numpy as np

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BASE = "/content/sample_data"
train_full = pd.read_parquet(f"{BASE}/train.parquet")
val_df     = pd.read_parquet(f"{BASE}/val.parquet")
test_df    = pd.read_parquet(f"{BASE}/test.parquet")

print("train:", train_full.shape)
print("val:  ", val_df.shape,  " <- checkpoint selection happens here, NOT on test")
print("test: ", test_df.shape, " <- read exactly once, at the very end")

print("\nTest set class distribution (left at the pool's real distribution, not capped):")
print((test_df["product"].value_counts(normalize=True) * 100).round(2).to_string())

# The splits are keyed on the canonical text so nothing straddles them. Re-assert it here:
# this notebook must not quietly be handed a leaky split.
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    ka = set(locals()[f"{a}_full" if a == "train" else f"{a}_df"]["key"])
    kb = set(locals()[f"{b}_full" if b == "train" else f"{b}_df"]["key"])
    assert not (ka & kb), f"canonical-key leak between {a} and {b}"
print("\ncanonical-key leakage across the three splits: none")

## Step 3 — Light text cleaning

Only the CFPB redaction masks (`XXXX`, `XX/XX/XXXX`) are removed — they carry no signal.

**Deliberately NOT done:** lowercasing, punctuation removal, stopword removal. DistilBERT's tokenizer handles those, and stripping them would destroy semantic cues the model relies on.

In [ ]:
import re

train_df = train_full.copy()

def light_clean(text):
    text = str(text)
    text = re.sub(r"X{2,}[/\-]X{2,}[/\-]X{2,}", " ", text)  # date masks XX/XX/XXXX
    text = re.sub(r"X{2,}[/\-]X{2,}", " ", text)             # partial date masks
    text = re.sub(r"\bX{2,}\b", " ", text)                   # standalone XXXX runs
    text = re.sub(r"\s+", " ", text).strip()                 # collapse whitespace
    return text

for d in (train_df, val_df, test_df):
    d["text"] = d["narrative"].apply(light_clean)

print("BEFORE:\n", train_df["narrative"].iloc[0][:280], "\n")
print("AFTER:\n",  train_df["text"].iloc[0][:280])

# Deliberately lighter than the Logistic Regression cleaner: digits, case and punctuation
# are kept, because a Transformer can use them. That asymmetry is a real difference between
# the two pipelines and is disclosed in the README rather than hidden.

## Step 4 — Encode labels

The label encoder is fitted on the **test set's** classes to guarantee all 10 are present, then applied to both.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Fit on TRAIN. The previous version fit on test_df, which is harmless only for as long as
# every class happens to appear in test -- and is the wrong habit for a project about leakage.
le = LabelEncoder()
le.fit(sorted(train_df["product"].unique()))

for d in (train_df, val_df, test_df):
    d["label"] = le.transform(d["product"])

NUM_LABELS  = len(le.classes_)
LABEL_NAMES = list(le.classes_)

assert sorted(val_df["product"].unique()) == LABEL_NAMES
assert sorted(test_df["product"].unique()) == LABEL_NAMES

print(f"{NUM_LABELS} classes (alphabetical -- same order as the LR notebook's `labels`):")
for i, name in enumerate(LABEL_NAMES):
    print(f"  {i}: {name}")

## Step 5 — Tokenize

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]],   preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]],  preserve_index=False)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_ds   = val_ds.map(tokenize,   batched=True, remove_columns=["text"])
test_ds  = test_ds.map(tokenize,  batched=True, remove_columns=["text"])

# How much of the corpus does MAX_LENGTH actually cut off? Quoted in the README as part of
# DistilBERT's cost, since TF-IDF reads every document in full.
lens = [len(tokenizer(t, truncation=False)["input_ids"]) for t in train_df["text"].head(5000)]
print(f"\ntruncated at {MAX_LENGTH} wordpieces: "
      f"{100 * np.mean(np.array(lens) > MAX_LENGTH):.1f}% of training documents "
      f"(median length {int(np.median(lens))})")

## Step 6 — Load the model and set up training

Metrics computed during training use **macro F1** as the headline, consistent with the Logistic Regression evaluation.

In [ ]:
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score, accuracy_score
import torch.nn as nn

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={i: n for i, n in enumerate(LABEL_NAMES)},
    label2id={n: i for i, n in enumerate(LABEL_NAMES)},
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":    accuracy_score(labels, preds),
        "macro_f1":    f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }

# Match the Logistic Regression side, which uses class_weight='balanced'. Without this the
# two models handle imbalance differently and the comparison measures that, not the models.
# Same formula sklearn uses: N / (K * n_k), computed on the capped training set.
counts = train_df["label"].value_counts().sort_index().values
class_weights = torch.tensor(len(train_df) / (NUM_LABELS * counts), dtype=torch.float)
print("class weights (sklearn 'balanced' equivalent):")
for i, w in enumerate(class_weights):
    print(f"  {w:.3f}  {LABEL_NAMES[i]}")

class WeightedTrainer(Trainer):
    # num_items_in_batch is required by recent transformers versions.
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=class_weights.to(outputs.logits.device))(
            outputs.logits.view(-1, NUM_LABELS), labels.view(-1)
        )
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="distilbert_complaints",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=RANDOM_STATE,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,      # VALIDATION, not test. The checkpoint is chosen here.
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

assert trainer.eval_dataset is val_ds, "checkpoint selection must not see the test set"
print("\nReady to train. Checkpoint selection on val; test is untouched until the end.")

## Step 7 — Train


In [ ]:
train_result = trainer.train()
print("\nTraining complete.")
print(train_result.metrics)

## Step 8 — Evaluate on the shared test set

This is the number that goes head-to-head against Logistic Regression.

In [ ]:
from sklearn.metrics import classification_report, balanced_accuracy_score

# FIRST AND ONLY read of the test set.
predictions = trainer.predict(test_ds)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

report = classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4)
print(report)

w = test_df["n_copies"].values   # intake-volume view

macro_f1    = f1_score(y_true, y_pred, average="macro")
weighted_f1 = f1_score(y_true, y_pred, average="weighted")
accuracy    = accuracy_score(y_true, y_pred)
bal_acc     = balanced_accuracy_score(y_true, y_pred)
macro_f1_w  = f1_score(y_true, y_pred, average="macro", sample_weight=w)
accuracy_w  = accuracy_score(y_true, y_pred, sample_weight=w)

print("=" * 62)
print("HEADLINE RESULTS -- DistilBERT")
print("=" * 62)
print(f"{'':<26}{'text diversity':>16}{'intake volume':>16}")
print(f"{'macro F1':<26}{macro_f1:>16.4f}{macro_f1_w:>16.4f}")
print(f"{'accuracy':<26}{accuracy:>16.4f}{accuracy_w:>16.4f}")
print(f"{'weighted F1':<26}{weighted_f1:>16.4f}")
print(f"{'balanced accuracy':<26}{bal_acc:>16.4f}")
print(f"\nTest set: {len(y_true):,} rows -- the same rows Logistic Regression was scored on.")

# Row-level predictions, so the LR notebook can run a paired bootstrap and McNemar's test.
pd.DataFrame({
    "key":        test_df["key"].values,
    "true":       le.inverse_transform(y_true),
    "prediction": le.inverse_transform(y_pred),
}).to_csv("distilbert_test_predictions.csv", index=False)
print("wrote distilbert_test_predictions.csv  -> Step 9 of the LR notebook")

## Step 9 — Confusion matrix

Shows which product categories get confused with which — this feeds the Business Insights section of the report.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SHORT = [n if len(n) <= 22 else n[:20] + "..." for n in LABEL_NAMES]

cm = confusion_matrix(y_true, y_pred)
# Row-normalised: supports now span ~512 to ~14,800, so raw counts render the rare-class
# rows invisible. Matches the LR notebook so the two figures can be read side by side.
cm_pct = 100 * cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(11, 9))
ConfusionMatrixDisplay(cm_pct, display_labels=SHORT).plot(
    ax=ax, cmap="Blues", xticks_rotation=45, values_format=".1f", colorbar=True
)
ax.set_title(f"DistilBERT -- % of each true class (macro F1 = {macro_f1:.3f})", fontsize=13, pad=15)
plt.tight_layout()
plt.savefig("confusion_matrix_distilbert.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved: confusion_matrix_distilbert.png  (row-normalised)")

## Step 10 — Save results and download

Saves everything needed for the report and the ML-vs-DL comparison table.

In [ ]:
with open("distilbert_results.txt", "w") as f:
    f.write("DistilBERT -- Consumer Complaint Classification\n")
    f.write("=" * 62 + "\n\n")
    f.write(f"Model              : {MODEL_NAME}\n")
    f.write(f"Training set       : {len(train_df):,} rows (capped -- a GPU budget)\n")
    f.write(f"Validation set     : {len(val_df):,} rows (checkpoint selection)\n")
    f.write(f"Test set           : {len(test_df):,} rows (read once)\n")
    f.write(f"Max token length   : {MAX_LENGTH}\n")
    f.write(f"Epochs             : {training_args.num_train_epochs}\n")
    f.write(f"Class-weighted loss: yes (matches LR's class_weight='balanced')\n")
    f.write(f"Training time      : {train_result.metrics['train_runtime']:.1f}s "
            f"on {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}\n\n")
    f.write(f"Macro F1          : {macro_f1:.4f}\n")
    f.write(f"Weighted F1       : {weighted_f1:.4f}\n")
    f.write(f"Accuracy          : {accuracy:.4f}\n")
    f.write(f"Balanced accuracy : {bal_acc:.4f}\n")
    f.write(f"Macro F1 (intake-volume weighted) : {macro_f1_w:.4f}\n\n")
    f.write("Per-class report:\n")
    f.write(report)

print(open("distilbert_results.txt").read())

from google.colab import files
files.download("distilbert_results.txt")
files.download("confusion_matrix_distilbert.png")
files.download("distilbert_test_predictions.csv")